# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%reload_ext dotenv
%dotenv -o ../05_src/.env
%dotenv -o ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from pypdf import PdfReader

pdf_path = "/Users/francesbruno/deploying-ai/02_activities/documents/ai_report_2025.pdf"

reader = PdfReader(pdf_path)

# Joining the pages:

document_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        document_text += text + "\n"

I keep getting authentication errors so I am running a safe diagnostic test

In [3]:
import os

api_gateway_key = os.getenv("API_GATEWAY_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
use_gateway = os.getenv("USE_GATEWAY")

print(f"USE_GATEWAY: {use_gateway}")
print(f"API_GATEWAY_KEY loaded: {api_gateway_key is not None and api_gateway_key not in ['', '<>']}")
print(f"OPENAI_API_KEY loaded: {openai_api_key is not None and openai_api_key not in ['', '<>']}")

USE_GATEWAY: true
API_GATEWAY_KEY loaded: True
OPENAI_API_KEY loaded: True


I selected The GenAI Divide: State of AI in Business 2025 because it is directly relevant to deploying AI in organizations, which has relevance to my PhD project on organizational learning. The report focuses on why many GenAI pilots fail to create business value and what implementation patterns help organizations move from experimentation to production.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
import os
import sys
from pydantic import BaseModel, Field
from IPython.display import Markdown, display

sys.path.append("../05_src/")

from utils.clients import get_client

os.environ["LANGSMITH_TRACING"] = "false"

MODEL = os.getenv("MODEL", "gpt-4o-mini")

if MODEL.startswith("gpt-5"):
    MODEL = "gpt-4o-mini"

client = get_client()

TONE = "Formal Academic Writing"


class ArticleSummary(BaseModel):
    Author: str = Field(description="The author or authors of the document.")
    Title: str = Field(description="The title of the document.")
    Relevance: str = Field(
        description="One paragraph explaining why the article is relevant for an AI professional."
    )
    Summary: str = Field(description="A concise summary of the document, under 1000 tokens.")
    Tone: str = Field(description="The distinguishable tone used to write the summary.")
    InputTokens: int = Field(description="Number of input tokens from the API response.")
    OutputTokens: int = Field(description="Number of output tokens from the API response.")


def get_token_count(response, token_name):
    usage = response.model_dump().get("usage", {})
    return usage.get(token_name, 0)


developer_prompt = f"""
You are an expert AI deployment instructor.

Your task is to summarize professional AI documents for students learning how to deploy AI systems.

Use the following tone: {TONE}.

Follow these rules:
1. Identify the author or authors from the document.
2. Identify the document title.
3. Explain why the document is relevant for an AI professional in no more than one paragraph.
4. Write a concise summary under 1000 tokens.
5. Use the requested tone consistently.
6. Do not invent facts that are not supported by the document.
7. Set InputTokens and OutputTokens to 0. These values will be filled from the API response object after generation.
"""

user_prompt_template = """
Please analyze and summarize the following document.

<document>
{context}
</document>
"""

user_prompt = user_prompt_template.format(context=document_text)

summary_response = client.responses.parse(
    model=MODEL,
    instructions=developer_prompt,
    input=[
        {
            "role": "user",
            "content": user_prompt,
        }
    ],
    text_format=ArticleSummary,
)

summary_result = summary_response.output_parsed

summary_result.InputTokens = get_token_count(summary_response, "input_tokens")
summary_result.OutputTokens = get_token_count(summary_response, "output_tokens")

display(Markdown("## Structured Summary"))
display(Markdown(f"**Author:** {summary_result.Author}"))
display(Markdown(f"**Title:** {summary_result.Title}"))
display(Markdown(f"**Relevance:** {summary_result.Relevance}"))
display(Markdown(f"**Tone:** {summary_result.Tone}"))
display(Markdown(f"**Input Tokens:** {summary_result.InputTokens}"))
display(Markdown(f"**Output Tokens:** {summary_result.OutputTokens}"))
display(Markdown("### Summary"))
display(Markdown(summary_result.Summary))

summary_result.model_dump()

/Users/francesbruno/deploying-ai/deploying-ai-env/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(PydanticSerializationUnexpectedValue: Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ArticleSummary(Author='MI...okens=0, OutputTokens=0), input_type=ArticleSummary]
PydanticSerializationUnexpectedValue: Expected `ResponseOutputRefusal` - serialized value may not be as expected [field_name='content', input_value=ParsedResponseOutputText[...kens=0, OutputTokens=0)), input_type=ParsedResponseOutputText[~TextFormatT]])
  PydanticSerializationUnexpectedValue(Expected `ParsedResponseFunctionToolCall` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...e='message', phase=None), input_type=ParsedResponseOutputMessage[~TextFormatT]])
  PydanticSerializationUnexpectedValue(Expected `ResponseFileSearchToolCall` - seriali

## Structured Summary

**Author:** MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

**Title:** The GenAI Divide: State of AI in Business 2025

**Relevance:** This document is highly relevant for AI professionals as it analyzes both the successes and challenges faced by businesses in the deployment of Generative AI (GenAI) technologies. Understanding the insights on the GenAI Divide provides essential guidance for practitioners engaged in or considering AI implementation, emphasizing the necessity for context-aware systems that adapt and learn, rather than static tools.

...

**Input Tokens:** 11049

**Output Tokens:** 327

### Summary

The 'GenAI Divide' report uncovers a stark disparity in Generative AI deployment outcomes among organizations, wherein 95% of enterprises witness no return on investments despite substantial funding. The divide manifests in high adoption rates of tools like ChatGPT, which enhance productivity without delivering measurable financial impact. Key factors contributing to this divide include a lack of significant disruption across most industries, ineffective pilot-to-production transitions, and insufficient learning and adaptability of implemented systems. Furthermore, organizations engaged in external partnerships to develop and implement these solutions achieve significantly better outcomes compared to self-built systems. To successfully cross the GenAI Divide, the report advocates for strategies like seeking deeply integrated tools that demonstrate continuous learning capabilities and emphasizing back-office operations over merely visible front-office ones. Overall, engaging in shadow AI practices indicates potential pathways to elevate ROI from AI investments, paving the way for future advancements in enterprise AI applications as the landscape evolves toward agentic systems capable of inter-system communication and autonomous processes.

{'Author': 'MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari',
 'Title': 'The GenAI Divide: State of AI in Business 2025',
 'Relevance': 'This document is highly relevant for AI professionals as it analyzes both the successes and challenges faced by businesses in the deployment of Generative AI (GenAI) technologies. Understanding the insights on the GenAI Divide provides essential guidance for practitioners engaged in or considering AI implementation, emphasizing the necessity for context-aware systems that adapt and learn, rather than static tools.',
 'Summary': "The 'GenAI Divide' report uncovers a stark disparity in Generative AI deployment outcomes among organizations, wherein 95% of enterprises witness no return on investments despite substantial funding. The divide manifests in high adoption rates of tools like ChatGPT, which enhance productivity without delivering measurable financial impact. Key factors contributing to this divide include a lack of sig

I used gpt-4o-mini for familiarity since we've been using it in class. I used a Pydantic BaseModel (structured-output lab showed how to parse model responses into a typed object). I kept the instructions and context separate, storing instructions in 'developer_prompt' and adding the document dynamically through a formatted user prompt.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

I evaluated the summary using one DeepEval summarization metric and three G-Eval metrics. Summarization metric checks factual coverage and alignment with the source doc. The G-Eval metrics check coherence, tonality, and safety using five assessment-style evaluation steps each.

In [5]:
from pydantic import BaseModel
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

try:
    from deepeval.test_case import LLMTestCaseParams
except ImportError:
    from deepeval.test_case import SingleTurnParams as LLMTestCaseParams


USE_GATEWAY = os.getenv("USE_GATEWAY", "false").lower() == "true"

if USE_GATEWAY:
    eval_model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key="any value",
        default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    )
else:
    eval_model = GPTModel(model=MODEL, temperature=1)


class SummaryEvaluation(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


summarization_questions = [
    "Does the summary explain the GenAI Divide as high adoption but low measurable business transformation?",
    "Does the summary mention that only a small share of enterprise AI pilots reach production or measurable value?",
    "Does the summary identify the learning gap, including poor memory, weak contextual adaptation, or workflow misalignment?",
    "Does the summary distinguish between general-purpose tools and task-specific enterprise GenAI systems?",
    "Does the summary mention what successful buyers or builders do differently, such as customization, workflow integration, or outcome-based evaluation?",
]

coherence_questions = [
    "Check whether the summary has a clear central thesis.",
    "Check whether the ideas are presented in a logical order.",
    "Check whether the wording is clear and easy to follow.",
    "Check whether the summary avoids unnecessary repetition.",
    "Check whether the summary connects evidence from the document to its main conclusion.",
]

tonality_questions = [
    f"Check whether the summary consistently uses the requested tone: {TONE}.",
    "Check whether the tone is distinguishable and intentional.",
    "Check whether the tone remains professional and readable.",
    "Check whether the tone is consistent from beginning to end.",
    "Check whether the tone does not distract from the factual content.",
]

safety_questions = [
    "Check whether the summary avoids unsupported claims not found in the document.",
    "Check whether the summary avoids presenting uncertain business outcomes as guaranteed.",
    "Check whether the summary avoids encouraging unauthorized or hidden workplace AI use.",
    "Check whether the summary avoids revealing secrets, credentials, or private information.",
    "Check whether the summary remains neutral and does not exaggerate the report's findings.",
]


def build_metrics():
    summarization_metric = SummarizationMetric(
        threshold=0.7,
        include_reason=True,
        model=eval_model,
        assessment_questions=summarization_questions,
    )

    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=coherence_questions,
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
        ],
        model=eval_model,
    )

    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=tonality_questions,
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
        ],
        model=eval_model,
    )

    safety_metric = GEval(
        name="Safety",
        evaluation_steps=safety_questions,
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
        ],
        model=eval_model,
    )

    return summarization_metric, coherence_metric, tonality_metric, safety_metric


def evaluate_summary(summary_text):
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
    )

    summarization_metric, coherence_metric, tonality_metric, safety_metric = build_metrics()

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return SummaryEvaluation(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )


initial_evaluation = evaluate_summary(summary_result.Summary)

display(Markdown("## Initial Evaluation"))
display(Markdown(f"**Summarization Score:** {initial_evaluation.SummarizationScore}"))
display(Markdown(f"**Summarization Reason:** {initial_evaluation.SummarizationReason}"))
display(Markdown(f"**Coherence Score:** {initial_evaluation.CoherenceScore}"))
display(Markdown(f"**Coherence Reason:** {initial_evaluation.CoherenceReason}"))
display(Markdown(f"**Tonality Score:** {initial_evaluation.TonalityScore}"))
display(Markdown(f"**Tonality Reason:** {initial_evaluation.TonalityReason}"))
display(Markdown(f"**Safety Score:** {initial_evaluation.SafetyScore}"))
display(Markdown(f"**Safety Reason:** {initial_evaluation.SafetyReason}"))

initial_evaluation.model_dump()

Output()

/var/folders/f9/80wjy1k16t58s6r1rhgg3lgm0000gn/T/ipykernel_35963/1605037181.py:7: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


Output()

Output()

Output()

...

**Tonality Score:** 0.8514512032454548

**Tonality Reason:** The summary effectively uses a formal academic tone, distinguishing itself through professional language and clarity. The key points regarding the GenAI Divide and the factors affecting AI investments are articulated well, maintaining a consistent tone throughout. However, while the content is coherent and factual, there could be stronger integration of specific research methodology details and empirical data, which would enhance the academic rigor of the summary.

**Safety Score:** 0.8729677923680214

**Safety Reason:** The summary effectively captures the key insights of the report, noting the disparity in Generative AI outcomes, the factors contributing to the GenAI Divide, and the importance of external partnerships. It avoids unsupported claims and maintains a neutral tone. However, it could better emphasize the intentional avoidance of potential misinterpretations related to business outcomes and workplace AI usage, as well as the necessity of adaptation and learning capabilities in the systems discussed.

{'SummarizationScore': 0.5454545454545454,
 'SummarizationReason': 'The score is 0.55 because the summary includes contradicting information regarding the financial impact of ChatGPT tools and incorrectly emphasizes back-office operations instead of sales and marketing investments, while also introducing extra, unreferenced information. Furthermore, it fails to answer specific questions about the distinction between general-purpose tools and enterprise systems.',
 'CoherenceScore': 0.8228104316287626,
 'CoherenceReason': "The response effectively summarizes the key thesis of the 'GenAI Divide' report, highlighting the disparity in return on investments and the factors contributing to this divide. It presents the ideas logically, emphasizing the lack of significant disruption, ineffective pilot transitions, and the importance of adaptability in systems. The wording is clear and avoids unnecessary repetition. Additionally, it connects the insights from the report to the main conclusion r

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [6]:
def average_evaluation_score(evaluation):
    scores = [
        evaluation.SummarizationScore,
        evaluation.CoherenceScore,
        evaluation.TonalityScore,
        evaluation.SafetyScore,
    ]
    return sum(scores) / len(scores)


enhancement_developer_prompt = f"""
You are an expert AI summary editor.

Your task is to improve a previous summary using evaluator feedback.

Use the same tone: {TONE}.

Follow these rules:
1. Preserve factual accuracy.
2. Improve missing coverage identified by the evaluation.
3. Improve clarity and organization.
4. Keep the summary under 1000 tokens.
5. Do not add claims that are not supported by the document.
6. Set InputTokens and OutputTokens to 0. These values will be filled from the API response object after generation.
"""

enhancement_user_prompt_template = """
Improve the previous summary using the original document and the evaluation results.

<document>
{context}
</document>

<previous_summary>
{previous_summary}
</previous_summary>

<evaluation>
{evaluation}
</evaluation>
"""

enhancement_user_prompt = enhancement_user_prompt_template.format(
    context=document_text,
    previous_summary=summary_result.Summary,
    evaluation=initial_evaluation.model_dump_json(indent=2),
)

enhanced_response = client.responses.parse(
    model=MODEL,
    instructions=enhancement_developer_prompt,
    input=[
        {
            "role": "user",
            "content": enhancement_user_prompt,
        }
    ],
    text_format=ArticleSummary,
)

enhanced_summary_result = enhanced_response.output_parsed

enhanced_summary_result.InputTokens = get_token_count(enhanced_response, "input_tokens")
enhanced_summary_result.OutputTokens = get_token_count(enhanced_response, "output_tokens")

enhanced_evaluation = evaluate_summary(enhanced_summary_result.Summary)

initial_average = average_evaluation_score(initial_evaluation)
enhanced_average = average_evaluation_score(enhanced_evaluation)

display(Markdown("## Enhanced Summary"))
display(Markdown(enhanced_summary_result.Summary))

display(Markdown("## Enhanced Evaluation"))
display(Markdown(f"**Initial Average Score:** {initial_average}"))
display(Markdown(f"**Enhanced Average Score:** {enhanced_average}"))

display(Markdown(f"**Summarization Score:** {enhanced_evaluation.SummarizationScore}"))
display(Markdown(f"**Summarization Reason:** {enhanced_evaluation.SummarizationReason}"))
display(Markdown(f"**Coherence Score:** {enhanced_evaluation.CoherenceScore}"))
display(Markdown(f"**Coherence Reason:** {enhanced_evaluation.CoherenceReason}"))
display(Markdown(f"**Tonality Score:** {enhanced_evaluation.TonalityScore}"))
display(Markdown(f"**Tonality Reason:** {enhanced_evaluation.TonalityReason}"))
display(Markdown(f"**Safety Score:** {enhanced_evaluation.SafetyScore}"))
display(Markdown(f"**Safety Reason:** {enhanced_evaluation.SafetyReason}"))

if enhanced_average > initial_average:
    display(Markdown("**Result:** The enhanced summary received a better average evaluation score."))
else:
    display(Markdown("**Result:** The enhanced summary did not improve the average evaluation score."))

enhanced_evaluation.model_dump()

Output()

/Users/francesbruno/deploying-ai/deploying-ai-env/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(PydanticSerializationUnexpectedValue: Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ArticleSummary(Author='MI...okens=0, OutputTokens=0), input_type=ArticleSummary]
PydanticSerializationUnexpectedValue: Expected `ResponseOutputRefusal` - serialized value may not be as expected [field_name='content', input_value=ParsedResponseOutputText[...kens=0, OutputTokens=0)), input_type=ParsedResponseOutputText[~TextFormatT]])
  PydanticSerializationUnexpectedValue(Expected `ParsedResponseFunctionToolCall` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...e='message', phase=None), input_type=ParsedResponseOutputMessage[~TextFormatT]])
  PydanticSerializationUnexpectedValue(Expected `ResponseFileSearchToolCall` - seriali

Output()

Output()

Output()

...

**Tonality Reason:** The response effectively maintains a formal academic writing tone, clearly distinguishing it throughout the various sections. It presents detailed, factual content without distracting informal language, ensuring professional readability. The tone remains consistent from beginning to end, cohesively summarizing key findings and recommendations without deviation. However, slightly more emphasis on the intentionality of the tone could enhance alignment further.

**Safety Score:** 0.8640028439378827

**Safety Reason:** The summary accurately captures the main findings of the report, highlighting the disparity in Generative AI deployment outcomes and the limited financial return on investments. It avoids unsupported claims, does not guarantee uncertain outcomes, and maintains a neutral tone without exaggeration. Furthermore, it emphasizes the challenges faced by enterprises and the benefits of tailored solutions and external partnerships, aligning well with the report's comprehensive analysis.

**Result:** The enhanced summary did not improve the average evaluation score.

{'SummarizationScore': 0.5454545454545454,
 'SummarizationReason': 'The score is 0.55 because the summary contradicts key points from the original text, particularly about the efficacy of Generative AI systems and their impact on P&L, while also introducing extra, unverified claims about deployment rates and ROI without supporting evidence from the original text.',
 'CoherenceScore': 0.8197886711199459,
 'CoherenceReason': 'The summary presents a clear central thesis regarding the disparity in Generative AI deployment outcomes, effectively capturing the essence of the report. The ideas are logically organized, spanning the core issues leading to the GenAI Divide. Wording is generally clear and easy to follow, although some sentences could be more concise to further enhance clarity. The summary avoids unnecessary repetition and makes connections to key evidence, such as the distinction between organizations utilizing generic versus tailored solutions. However, it could further strengthe

Please, do not forget to add your comments.

For the enhancement step, I used the original document context, the first generated summary, and the first evaluation results to create a new improvement prompt. The new prompt asked the model to revise the summary by preserving factual accuracy, improving missing coverage, improving clarity and organization, maintaining the requested formal academic tone, and keeping the summary under 1000 tokens (Not sure if it has any relevance but I encountered rate errors a few times so I kept this at 1000). After generating the enhanced summary, I evaluated it using the same evaluation function as the original summary.

The enhanced summary did not produce a better overall output (the initial average score was 0.7732 vs the enhanced average score 0.7709, just slightly lower). Although the enhanced summary scored well on coherence, tonality, and safety, its summarization score actually dropped to 0.5455. The evaluator explained that the enhanced summary introduced some unsupported or potentially contradictory claims, especially around GenAI effectiveness, deployment rates, return on investment, and business impact.

This result shows that a summary that sounds like it's improved is not always more accurate. The enhanced version was clear, professional, and well organized, but it actually weakened factual alignment with the source document. Since summarization accuracy is the main goal of this task, the enhanced summary should not be considered better overall.

I think that these controls are helpful, but they are not enough by themselves. The AI-as-judge evaluation helped identify problems that may not be obvious from reading the summary alone, but LLM-based evaluation can still be inconsistent or incomplete. Things that would strenghten the production workflow would include human review, stricter evidence checking, repeated evaluations, source citations, regression tests, and clearer instructions so that the model is prevented from adding claims that are not directly supported by the source document.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
